# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E2: Directed Functional Connectivity (Cross-Correlations + Granger Causality)

Using the 10 Hz ΔF/F data from zarr stores, we now have proper time-series for:
- **E2.1** Sub-second cross-correlations between cell pairs at fine temporal lags
- **E2.2** Granger causality on within-trial 10 Hz traces
- **E2.3** Directed connection motifs across cell-type pairs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E2.1  Sub-second cross-correlations from 10 Hz ΔF/F traces
# ══════════════════════════════════════════════════════════════════════

# Load from the same mouse used in E1
pk = load_zarr_10hz(mouse_pick)
dff_10hz = pk['dff']           # (n_trials, 41, n_cells)
time_rel_e2 = pk['time_rel']
ti_10hz = pk['trial_info']
uids_10hz = pk['unique_ids']

# Match cells to the subsampled set from E1 (or re-subsample)
obs_mouse_e2 = obs[obs['mouse_id'] == mouse_pick]
cell_map_e2 = {}
for ci, uid in enumerate(uids_10hz):
    m = obs_mouse_e2[obs_mouse_e2['unique_id'] == uid]
    if len(m) > 0:
        cell_map_e2[ci] = m.iloc[0]['subclass_name']

matched_e2 = sorted(cell_map_e2.keys())
sc_e2 = np.array([cell_map_e2[ci] for ci in matched_e2])

# Subsample for tractability
np.random.seed(42)
n_sub_e2 = min(80, len(matched_e2))
sub_idx = np.sort(np.random.choice(len(matched_e2), n_sub_e2, replace=False))
sub_cells = [matched_e2[i] for i in sub_idx]
sub_sc = sc_e2[sub_idx]

# Use stimulus period from non-gray trials
non_gray_e2 = ~ti_10hz['gray_screen'].astype(bool)
grating_idx_e2 = np.where(non_gray_e2.values)[0]
stim_mask_e2 = (time_rel_e2 >= 0) & (time_rel_e2 <= 2.0)
stim_tp_e2 = np.where(stim_mask_e2)[0]

# Concatenate stimulus-period traces across grating trials → pseudo-continuous
n_gt = len(grating_idx_e2)
dff_concat = np.zeros((n_gt * len(stim_tp_e2), n_sub_e2))
for ci_sub, ci in enumerate(sub_cells):
    traces = dff_10hz[grating_idx_e2][:, stim_tp_e2, ci]
    dff_concat[:, ci_sub] = traces.ravel()

print(f"Concatenated traces: {dff_concat.shape} (timepoints × cells) at 10 Hz")
print(f"  = {n_gt} trials × {len(stim_tp_e2)} timepoints/trial = {dff_concat.shape[0]} total")

# ── Compute pairwise cross-correlation at sub-second lags ──
max_lag_e2 = 5  # ±5 samples = ±500 ms
lags_e2 = np.arange(-max_lag_e2, max_lag_e2 + 1) * 0.1

# Compute for all pairs
n_pairs_e2 = n_sub_e2 * (n_sub_e2 - 1) // 2
xcorr_matrix = np.zeros((n_pairs_e2, len(lags_e2)))
pair_info = []
pi = 0
for i in range(n_sub_e2):
    for j in range(i+1, n_sub_e2):
        xcorr_matrix[pi] = xcorr_pair(dff_concat[:, i], dff_concat[:, j], max_lag_e2)
        pair_info.append({'i': i, 'j': j, 'sc_i': sub_sc[i], 'sc_j': sub_sc[j]})
        pi += 1

print(f"Cross-correlations computed for {n_pairs_e2} pairs")

# ── Visualization: mean CCG by pair type ──
pair_df = pd.DataFrame(pair_info)
pair_df['pair_type'] = pair_df.apply(
    lambda r: f"{SUBCLASS_SHORT.get(r['sc_i'], '?')}–{SUBCLASS_SHORT.get(r['sc_j'], '?')}", axis=1)

# Group: E-E, E-I, I-I
def ei_category(sc_i, sc_j):
    ei = 'Glut' in sc_i
    ej = 'Glut' in sc_j
    if ei and ej: return 'E–E'
    if not ei and not ej: return 'I–I'
    return 'E–I'

pair_df['ei_cat'] = [ei_category(r['sc_i'], r['sc_j']) for _, r in pair_df.iterrows()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean CCG by E-E, E-I, I-I
ax = axes[0]
for cat, color in [('E–E', 'green'), ('E–I', 'orange'), ('I–I', 'red')]:
    mask = pair_df['ei_cat'] == cat
    if mask.sum() == 0:
        continue
    mean_cc = np.mean(xcorr_matrix[mask.values], axis=0)
    sem_cc = np.std(xcorr_matrix[mask.values], axis=0) / np.sqrt(mask.sum())
    ax.fill_between(lags_e2, mean_cc - sem_cc, mean_cc + sem_cc, alpha=0.2, color=color)
    ax.plot(lags_e2, mean_cc, color=color, linewidth=2, label=f'{cat} (n={mask.sum()})')
ax.axvline(0, color='k', ls='--', alpha=0.4)
ax.set_xlabel('Lag (s)')
ax.set_ylabel('Correlation')
ax.set_title('E2: Cross-Correlogram by E/I Category', fontweight='bold')
ax.legend(fontsize=9)

# Peak lag distribution
ax = axes[1]
peak_lags_e2 = lags_e2[np.argmax(xcorr_matrix, axis=1)]
for cat, color in [('E–E', 'green'), ('E–I', 'orange'), ('I–I', 'red')]:
    mask = pair_df['ei_cat'] == cat
    ax.hist(peak_lags_e2[mask.values], bins=len(lags_e2), alpha=0.5, color=color,
            label=cat, density=True)
ax.set_xlabel('Peak Lag (s)')
ax.set_ylabel('Density')
ax.set_title('E2: Peak CCG Lag Distribution', fontweight='bold')
ax.legend(fontsize=9)

# Zero-lag vs peak correlation
ax = axes[2]
zero_lag_idx = max_lag_e2
zero_corr = xcorr_matrix[:, zero_lag_idx]
peak_corr_e2 = np.max(xcorr_matrix, axis=1)
ax.scatter(zero_corr, peak_corr_e2, alpha=0.3, s=10, c='steelblue')
ax.plot([0, 0.5], [0, 0.5], 'k--', alpha=0.3)
ax.set_xlabel('Zero-Lag Correlation')
ax.set_ylabel('Peak Correlation')
ax.set_title('E2: Zero-Lag vs Peak Correlation', fontweight='bold')

plt.suptitle('E2: Sub-Second Cross-Correlations (10 Hz)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E2.2  Granger Causality on 10 Hz traces + directed motif analysis
# ══════════════════════════════════════════════════════════════════════

# Use concatenated 10 Hz traces (already computed above)
# Further subsample for GC (computationally expensive)
n_gc = min(50, n_sub_e2)
gc_cell_idx = np.sort(np.random.choice(n_sub_e2, n_gc, replace=False))
gc_sc = sub_sc[gc_cell_idx]

print(f"Computing pairwise Granger causality for {n_gc} cells...")
gc_results = []
pair_count = 0
n_gc_pairs = n_gc * (n_gc - 1) // 2

for i in range(n_gc):
    for j in range(i+1, n_gc):
        xi = dff_concat[:, gc_cell_idx[i]]
        xj = dff_concat[:, gc_cell_idx[j]]
        # Remove NaN segments
        valid = ~np.isnan(xi) & ~np.isnan(xj)
        if valid.sum() < 50:
            continue
        res = pairwise_granger(xi[valid], xj[valid], max_lag=3)
        gc_results.append({
            'i': i, 'j': j,
            'sc_i': gc_sc[i], 'sc_j': gc_sc[j],
            'p_j_to_i': res['y_to_x_p'],
            'p_i_to_j': res['x_to_y_p'],
        })
        pair_count += 1
        if pair_count % 200 == 0:
            print(f"  {pair_count}/{n_gc_pairs} pairs...")

gc_df = pd.DataFrame(gc_results)
gc_df['sig_j_to_i'] = gc_df['p_j_to_i'] < 0.01
gc_df['sig_i_to_j'] = gc_df['p_i_to_j'] < 0.01
gc_df['sig_any'] = gc_df['sig_j_to_i'] | gc_df['sig_i_to_j']

total_edges = gc_df['sig_j_to_i'].sum() + gc_df['sig_i_to_j'].sum()
print(f"\nSignificant directed edges: {total_edges} out of {2 * len(gc_df)} possible "
      f"({total_edges/(2*len(gc_df))*100:.1f}%)")

# ── Directed connection probability matrix: subclass × subclass ──
present_sc_gc = [s for s in present_subclasses if s in gc_sc]
n_sgc = len(present_sc_gc)
directed_prob = np.zeros((n_sgc, n_sgc))
directed_count = np.zeros((n_sgc, n_sgc))

for _, row in gc_df.iterrows():
    if row['sc_i'] not in present_sc_gc or row['sc_j'] not in present_sc_gc:
        continue
    si = present_sc_gc.index(row['sc_i'])
    sj = present_sc_gc.index(row['sc_j'])
    directed_count[si, sj] += 1
    directed_count[sj, si] += 1
    if row['sig_i_to_j']:
        directed_prob[si, sj] += 1
    if row['sig_j_to_i']:
        directed_prob[sj, si] += 1

directed_frac = directed_prob / np.maximum(directed_count, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
s_labels_gc = [SUBCLASS_SHORT.get(s, s) for s in present_sc_gc]

ax = axes[0]
sns.heatmap(directed_frac, annot=True, fmt='.3f', xticklabels=s_labels_gc, yticklabels=s_labels_gc,
            cmap='Reds', ax=ax, vmin=0)
ax.set_xlabel('To')
ax.set_ylabel('From')
ax.set_title('E2: Directed GC Connection Probability\n(from row → to column)', fontweight='bold')

# Asymmetry
ax = axes[1]
asym = np.zeros_like(directed_frac)
for i in range(n_sgc):
    for j in range(n_sgc):
        denom = directed_frac[i, j] + directed_frac[j, i]
        if denom > 0:
            asym[i, j] = (directed_frac[i, j] - directed_frac[j, i]) / denom

sns.heatmap(asym, annot=True, fmt='.2f', xticklabels=s_labels_gc, yticklabels=s_labels_gc,
            cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_xlabel('To')
ax.set_ylabel('From')
ax.set_title('E2: Connectivity Asymmetry Index\n(+ = net flow From→To)', fontweight='bold')

plt.suptitle('E2: Granger Causality on 10 Hz ΔF/F', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── GC-significant connection probability vs distance ──
# Re-derive coordinates for the GC-subsampled cells
gc_cell_global = [matched_e2[gc_cell_idx[i]] for i in range(n_gc)]
gc_uids = [uids_10hz[ci] for ci in gc_cell_global]
gc_coords = np.zeros((n_gc, 3))
for i, uid in enumerate(gc_uids):
    row = obs_mouse_e2[obs_mouse_e2['unique_id'] == uid]
    if len(row) > 0:
        gc_coords[i] = [row.iloc[0]['x_loc'], row.iloc[0]['y_loc'], row.iloc[0].get('z_loc', 0)]

gc_df['dist'] = [np.linalg.norm(gc_coords[r['i']] - gc_coords[r['j']]) for _, r in gc_df.iterrows()]

fig, ax = plt.subplots(figsize=(8, 5))
dist_bins = np.arange(0, 400, 50)
dist_centers = (dist_bins[:-1] + dist_bins[1:]) / 2
prob_vs_dist = []
for b in range(len(dist_bins)-1):
    bm = (gc_df['dist'] >= dist_bins[b]) & (gc_df['dist'] < dist_bins[b+1])
    if bm.sum() > 5:
        prob_vs_dist.append(gc_df.loc[bm, 'sig_any'].mean())
    else:
        prob_vs_dist.append(np.nan)

ax.plot(dist_centers, prob_vs_dist, 'ko-', linewidth=2, markersize=6)
ax.set_xlabel('Pairwise Distance (µm)')
ax.set_ylabel('P(significant GC connection)')
ax.set_title('E2: Directed Connection Probability vs Distance', fontweight='bold')
plt.tight_layout()
plt.show()